# anatid in ten minutes

anatid is an embedded, bitemporal graph memory for AI agents: one DuckDB file that holds facts,
the entities they are about, the edges between those entities, the raw evidence each fact came
from, and every correction ever made. Retrieval runs three ways at once, by text, by vector and by
walking the graph, and fuses the three.

This notebook needs no API key, no server and no network: the embeddings come from
`HashEmbedder`, a deterministic bag-of-words stand-in, and the database lives in memory.

```
pip install anatid
```

In [1]:
from datetime import datetime, timedelta

from anatid import Anatid, HashEmbedder

DIM = 64
db = Anatid.open(":memory:", tenant=1, embedding_dim=DIM, embedder=HashEmbedder(dim=DIM))
info = db.info()
print(f"anatid {__import__('anatid').__version__}, schema v{info.schema_version}, duckdb {info.duckdb_version}")

# A fixed clock, so every run of this notebook prints the same dates.
t0 = datetime(2026, 3, 2, 9, 0)

anatid 0.4.1, schema v4, duckdb 1.5.5


## 1. Evidence before belief

anatid stores the source text first, as an *episode*, and files every memory derived from it
under that episode's id. That is what `provenance()` walks back to later: not a citation the model
made up, but the note the fact was read from.

In [2]:
note = db.episode(
    "Standup 2026-03-02: Ada leads the Kestrel team. Kestrel owns the ingest service, "
    "and Bo maintains it day to day. Ada takes her coffee as a dark roast.",
    source="notes/2026-03-02.md", writer="notes-bot", now=t0,
)
print(f"episode {note.episode_id} from {note.source!r}, written by {note.writer}")

episode 884810358250598400 from 'notes/2026-03-02.md', written by notes-bot


## 2. Facts, and the entities they are about

A memory is one durable statement. `entities=` creates the named entities on demand and draws an
`ABOUT` edge from the memory to each of them; that edge is what makes the memory reachable from
the graph side. `kind` is a free label (`fact`, `preference`, `constraint`, `decision`, ...). With
an embedder on the handle, every memory is embedded as it is written.

In [3]:
leads = db.remember("Ada leads Kestrel", entities=["Ada", "Kestrel"],
                    writer="notes-bot", episode_id=note.episode_id, now=t0)
owns = db.remember("Kestrel owns the ingest service", entities=["Kestrel", "ingest service"],
                   writer="notes-bot", episode_id=note.episode_id, now=t0)
maintains = db.remember("Bo maintains the ingest service", entities=["Bo", "ingest service"],
                        writer="notes-bot", episode_id=note.episode_id, now=t0)
coffee = db.remember("Ada prefers dark roast coffee", entities=["Ada", "coffee"], kind="preference",
                     writer="notes-bot", episode_id=note.episode_id, now=t0)

for m in (leads, owns, maintains, coffee):
    about = [e.name for e in db.entities_of(m.memory_id)]
    print(f"{m.kind:<10} {m.content!r:<40} about {about}")

fact       'Ada leads Kestrel'                      about ['Ada', 'Kestrel']
fact       'Kestrel owns the ingest service'        about ['Kestrel', 'ingest service']
fact       'Bo maintains the ingest service'        about ['ingest service', 'Bo']
preference 'Ada prefers dark roast coffee'          about ['Ada', 'coffee']


## 3. Relations: the hops recall walks

`ABOUT` edges connect memories to entities. `RELATES_TO` edges connect entities to each other, and
they are the hops that graph recall traverses: two hops out from Ada is Ada → Kestrel → ingest
service, which reaches Bo's memory even though no fact about Bo mentions Ada.

In [4]:
db.relate("Ada", "Kestrel", rel_kind="leads", writer="notes-bot", now=t0)
db.relate("Kestrel", "ingest service", rel_kind="owns", writer="notes-bot", now=t0)
db.relate("Bo", "ingest service", rel_kind="maintains", writer="notes-bot", now=t0)

s = db.stats()
print({k: s[k] for k in ("episodes", "memories", "entities", "edges_about", "edges_relates")})
print("two hops from Ada:", [m.content for m in db.recall_2hop("Ada", limit=10)])

{'episodes': 1, 'memories': 4, 'entities': 5, 'edges_about': 8, 'edges_relates': 3}
two hops from Ada: ['Ada prefers dark roast coffee', 'Bo maintains the ingest service', 'Kestrel owns the ingest service', 'Ada leads Kestrel']


## 4. Recall: three arms, one ranking

`recall(query)` runs every arm that has input. The text arm is BM25 over the memories' content;
the vector arm is cosine over their embeddings (the handle embeds the query for you); the graph
arm seeds itself from the entity names it finds in the query and expands two hops. The three lists
are fused with weighted reciprocal rank fusion, and the result says which arms ran, from which
seeds, and with which weights.

In [5]:
hits = db.recall("who maintains the ingest service", k=5)
print(f"arms={hits.arms} seeds={hits.seeds} weights={hits.weights}\n")
for h in hits:
    print(f"[{h.rank}] {h.score:.4f} {h.content!r:<40} via {'+'.join(h.sources)}")

arms=('vector', 'text', 'graph') seeds=('ingest service',) weights={'vector': 1.0, 'text': 0.25, 'graph': 0.5}

[1] 0.0287 'Bo maintains the ingest service'        via vector+text+graph
[2] 0.0282 'Kestrel owns the ingest service'        via vector+text+graph
[3] 0.0237 'Ada leads Kestrel'                      via vector+graph
[4] 0.0236 'Ada prefers dark roast coffee'          via vector+graph


## 5. Correction, not overwrite

When a fact changes, the old memory is *superseded*: it is closed with a `valid_to`, the new one
opens, and a `SUPERSEDES` edge joins them. Nothing is deleted. `correct()` does that and moves the
graph edges the change implies in the same transaction.

In [6]:
t1 = t0 + timedelta(days=105)  # 2026-06-15: Cy takes the ingest service over from Bo
receipt = db.correct(
    maintains.memory_id, "Cy maintains the ingest service", entities=["Cy", "ingest service"],
    add_relations=[("Cy", "ingest service", "maintains")],
    remove_relations=[("Bo", "ingest service", "maintains")],
    writer="notes-bot", now=t1,
)
old, new = db.get(maintains.memory_id), receipt.new
print(f"old: {old.content!r} is_current={old.is_current} valid {old.valid_from.date()} to {old.valid_to.date()}")
print(f"new: {new.content!r} is_current={new.is_current} valid from {new.valid_from.date()}")
print(f"edges opened: {len(receipt.opened)}, closed: {receipt.edges_closed}")

old: 'Bo maintains the ingest service' is_current=False valid 2026-03-02 to 2026-06-15
new: 'Cy maintains the ingest service' is_current=True valid from 2026-06-15
edges opened: 1, closed: 1


## 6. Time travel

anatid is bitemporal: every row carries valid time (when the fact was true) and transaction time
(when the database learned it). `as_of(t)` scopes any read to what was believed at `t`. There is
no rewinding; it is a filter over those columns.

In [7]:
april = t0 + timedelta(days=30)
print("as of April: ", [m.content for m in db.as_of(april).context("ingest service")])
print("today:       ", [m.content for m in db.context("ingest service")])

as of April:  ['Bo maintains the ingest service', 'Kestrel owns the ingest service']
today:        ['Cy maintains the ingest service', 'Kestrel owns the ingest service']


## 7. Provenance

From any memory, walk the chain of corrections back to the first assertion, and from there to the
notes each version was read from.

In [8]:
prov = db.provenance(new.memory_id)
print(f"depth {prov.depth}, writers {list(prov.writers)}")
for link in prov.chain:
    state = "current" if link.is_current else "closed "
    print(f"  {state} {link.content!r} (by {link.writer}, from {link.valid_from.date()})")
print("evidence:", prov.source_text[:70] + "...")

depth 1, writers ['notes-bot']
  current 'Cy maintains the ingest service' (by notes-bot, from 2026-06-15)
  closed  'Bo maintains the ingest service' (by notes-bot, from 2026-03-02)
evidence: Standup 2026-03-02: Ada leads the Kestrel team. Kestrel owns the inges...


## 8. Forgetting

`forget()` closes a memory (soft), or removes the row, its edges, its embedding and its audit
trail (`hard=True`), for an erasure request. `prune()` applies an age or usage policy to many
memories at once, and reports what it would remove before it removes anything.

In [9]:
soft = db.forget(coffee.memory_id, reason="preference withdrawn", writer="notes-bot", now=t1)
print("soft forget: current =", db.get(coffee.memory_id).is_current, "| hard =", soft.hard)

report = db.prune(older_than=t0 + timedelta(days=1), dry_run=True)
print("prune (dry run) would touch", len(report.memory_ids), "memories written before day 2")

hard = db.forget(coffee.memory_id, hard=True, reason="erasure request", writer="notes-bot", now=t1)
print("hard forget: rows removed =", hard.rows_removed, "| still there? ", db.get(coffee.memory_id) is not None)

soft forget: current = False | hard = False
prune (dry run) would touch 2 memories written before day 2
hard forget: rows removed = 9 | still there?  False


## 9. It is one file, and it is SQL

Everything above is rows in DuckDB tables. `stats()` counts them, `doctor()` checks their
integrity, and `execute()` runs any read-only query you like against the schema documented in
[`docs/architecture.md`](../../docs/architecture.md).

In [10]:
s = db.stats()
print({k: s[k] for k in ("memories", "current_memories", "entities", "edges_relates", "edges_supersedes")})
print("doctor ok:", db.doctor().ok)
rows = db.execute(
    "SELECT content, valid_from::DATE, valid_to::DATE FROM memories WHERE tenant_id = 1 ORDER BY valid_from, memory_id"
).fetchall()
for content, vf, vt in rows:
    print(f"  {vf} to {str(vt or 'now'):<10} {content}")
db.close()

{'memories': 4, 'current_memories': 3, 'entities': 6, 'edges_relates': 4, 'edges_supersedes': 1}
doctor ok: True
  2026-03-02 to now        Ada leads Kestrel
  2026-03-02 to now        Kestrel owns the ingest service
  2026-03-02 to now        Bo maintains the ingest service
  2026-03-02 to 2026-06-15 Bo maintains the ingest service
  2026-06-15 to now        Cy maintains the ingest service


## Where next

- [`02_ingest_notes.ipynb`](02_ingest_notes.ipynb): let a model turn whole notes into reviewed
  memory patches instead of writing one fact at a time.
- [`03_retrieval.ipynb`](03_retrieval.ipynb): what each recall arm sees, how the fusion weighs
  them, and how to tune it.
- [`04_agents_and_mcp.ipynb`](04_agents_and_mcp.ipynb): the same memory behind an OpenAI Agents
  SDK agent and behind an MCP server.
- The README and `docs/` for the file format, the server profile and the benchmarks.